# Complex: CloudFormation + Lambda + API Gateway Polly Pipeline

This notebook deploys the full infrastructure (IAM role, Lambda, API Gateway) using CloudFormation, then deploys the Lambda code and invokes the endpoint.

```
CloudFormation deploy  →  Lambda + API Gateway  →  POST /synthesize  →  S3
```

> Running inside SageMaker — credentials come from the instance role automatically.

**Required IAM permissions on your SageMaker execution role:**
- `cloudformation:*` (create/update/describe stacks)
- `lambda:*`, `apigateway:*`, `iam:*` (for CloudFormation to provision resources)
- `s3:PutObject` on your target bucket

## 1. Configuration

In [ ]:
# ── Set these before running ─────────────────────────────────────────────────
AWS_REGION     = "us-east-1"
S3_BUCKET_NAME = "your-bucket-name"   # <-- replace
BETA_STACK     = "PollyBetaStack"
PROD_STACK     = "PollyProdStack"
# ─────────────────────────────────────────────────────────────────────────────

## 2. CloudFormation Templates

Inline versions of `template-beta.yml` and `template-prod.yml`.

In [ ]:
TEMPLATE_BETA = """
AWSTemplateFormatVersion: '2010-09-09'
Description: Polly TTS Beta Stack - Lambda + API Gateway + IAM Role
Parameters:
  S3BucketName:
    Type: String
Resources:
  BetaExecutionRole:
    Type: AWS::IAM::Role
    Properties:
      RoleName: PollyBetaExecutionRole
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: lambda.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: PollyBetaPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action: polly:SynthesizeSpeech
                Resource: "*"
              - Effect: Allow
                Action: s3:PutObject
                Resource: !Sub "arn:aws:s3:::${S3BucketName}/polly-audio/beta/*"
              - Effect: Deny
                Action: s3:PutObject
                Resource: !Sub "arn:aws:s3:::${S3BucketName}/polly-audio/prod/*"
              - Effect: Allow
                Action: [logs:CreateLogGroup, logs:CreateLogStream, logs:PutLogEvents]
                Resource: "*"
  BetaLambdaFunction:
    Type: AWS::Lambda::Function
    Properties:
      FunctionName: PollyTextToSpeech_Beta
      Runtime: python3.11
      Handler: handler.lambda_handler
      Role: !GetAtt BetaExecutionRole.Arn
      Code:
        ZipFile: |
          def lambda_handler(event, context):
              return {"statusCode": 200, "body": "placeholder"}
      Environment:
        Variables:
          ENVIRONMENT: beta
          S3_BUCKET_NAME: !Ref S3BucketName
  BetaApiGateway:
    Type: AWS::ApiGateway::RestApi
    Properties:
      Name: PollyBetaAPI
  BetaResource:
    Type: AWS::ApiGateway::Resource
    Properties:
      RestApiId: !Ref BetaApiGateway
      ParentId: !GetAtt BetaApiGateway.RootResourceId
      PathPart: synthesize
  BetaMethod:
    Type: AWS::ApiGateway::Method
    Properties:
      RestApiId: !Ref BetaApiGateway
      ResourceId: !Ref BetaResource
      HttpMethod: POST
      AuthorizationType: NONE
      Integration:
        Type: AWS_PROXY
        IntegrationHttpMethod: POST
        Uri: !Sub
          - arn:aws:apigateway:${AWS::Region}:lambda:path/2015-03-31/functions/${LambdaArn}/invocations
          - LambdaArn: !GetAtt BetaLambdaFunction.Arn
  BetaLambdaPermission:
    Type: AWS::Lambda::Permission
    Properties:
      FunctionName: !GetAtt BetaLambdaFunction.Arn
      Action: lambda:InvokeFunction
      Principal: apigateway.amazonaws.com
      SourceArn: !Sub "arn:aws:execute-api:${AWS::Region}:${AWS::AccountId}:${BetaApiGateway}/*/POST/synthesize"
  BetaDeployment:
    Type: AWS::ApiGateway::Deployment
    DependsOn: BetaMethod
    Properties:
      RestApiId: !Ref BetaApiGateway
      StageName: beta
Outputs:
  LambdaArn:
    Value: !GetAtt BetaLambdaFunction.Arn
  ApiGatewayUrl:
    Value: !Sub "https://${BetaApiGateway}.execute-api.${AWS::Region}.amazonaws.com/beta/synthesize"
"""

In [ ]:
TEMPLATE_PROD = """
AWSTemplateFormatVersion: '2010-09-09'
Description: Polly TTS Prod Stack - Lambda + API Gateway + IAM Role
Parameters:
  S3BucketName:
    Type: String
Resources:
  ProdExecutionRole:
    Type: AWS::IAM::Role
    Properties:
      RoleName: PollyProdExecutionRole
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: lambda.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: PollyProdPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action: polly:SynthesizeSpeech
                Resource: "*"
              - Effect: Allow
                Action: s3:PutObject
                Resource: !Sub "arn:aws:s3:::${S3BucketName}/polly-audio/prod/*"
              - Effect: Allow
                Action: [logs:CreateLogGroup, logs:CreateLogStream, logs:PutLogEvents]
                Resource: "*"
  ProdLambdaFunction:
    Type: AWS::Lambda::Function
    Properties:
      FunctionName: PollyTextToSpeech_Prod
      Runtime: python3.11
      Handler: handler.lambda_handler
      Role: !GetAtt ProdExecutionRole.Arn
      Code:
        ZipFile: |
          def lambda_handler(event, context):
              return {"statusCode": 200, "body": "placeholder"}
      Environment:
        Variables:
          ENVIRONMENT: prod
          S3_BUCKET_NAME: !Ref S3BucketName
  ProdApiGateway:
    Type: AWS::ApiGateway::RestApi
    Properties:
      Name: PollyProdAPI
  ProdResource:
    Type: AWS::ApiGateway::Resource
    Properties:
      RestApiId: !Ref ProdApiGateway
      ParentId: !GetAtt ProdApiGateway.RootResourceId
      PathPart: synthesize
  ProdMethod:
    Type: AWS::ApiGateway::Method
    Properties:
      RestApiId: !Ref ProdApiGateway
      ResourceId: !Ref ProdResource
      HttpMethod: POST
      AuthorizationType: NONE
      Integration:
        Type: AWS_PROXY
        IntegrationHttpMethod: POST
        Uri: !Sub
          - arn:aws:apigateway:${AWS::Region}:lambda:path/2015-03-31/functions/${LambdaArn}/invocations
          - LambdaArn: !GetAtt ProdLambdaFunction.Arn
  ProdLambdaPermission:
    Type: AWS::Lambda::Permission
    Properties:
      FunctionName: !GetAtt ProdLambdaFunction.Arn
      Action: lambda:InvokeFunction
      Principal: apigateway.amazonaws.com
      SourceArn: !Sub "arn:aws:execute-api:${AWS::Region}:${AWS::AccountId}:${ProdApiGateway}/*/POST/synthesize"
  ProdDeployment:
    Type: AWS::ApiGateway::Deployment
    DependsOn: ProdMethod
    Properties:
      RestApiId: !Ref ProdApiGateway
      StageName: prod
Outputs:
  LambdaArn:
    Value: !GetAtt ProdLambdaFunction.Arn
  ApiGatewayUrl:
    Value: !Sub "https://${ProdApiGateway}.execute-api.${AWS::Region}.amazonaws.com/prod/synthesize"
"""

## 3. Deploy CloudFormation Stacks

In [ ]:
import boto3, time

cfn = boto3.client("cloudformation", region_name=AWS_REGION)

def deploy_stack(stack_name: str, template_body: str):
    params = [{"ParameterKey": "S3BucketName", "ParameterValue": S3_BUCKET_NAME}]
    caps   = ["CAPABILITY_NAMED_IAM"]
    try:
        cfn.describe_stacks(StackName=stack_name)
        print(f"Updating {stack_name}...")
        cfn.update_stack(StackName=stack_name, TemplateBody=template_body, Parameters=params, Capabilities=caps)
        waiter = cfn.get_waiter("stack_update_complete")
    except cfn.exceptions.ClientError as e:
        if "does not exist" in str(e):
            print(f"Creating {stack_name}...")
            cfn.create_stack(StackName=stack_name, TemplateBody=template_body, Parameters=params, Capabilities=caps)
            waiter = cfn.get_waiter("stack_create_complete")
        elif "No updates are to be performed" in str(e):
            print(f"{stack_name}: already up to date")
            return
        else:
            raise
    waiter.wait(StackName=stack_name)
    print(f"{stack_name}: deployed successfully")

deploy_stack(BETA_STACK, TEMPLATE_BETA)
deploy_stack(PROD_STACK, TEMPLATE_PROD)

## 4. Get Stack Outputs (API Gateway URLs)

In [ ]:
def get_output(stack_name: str, key: str) -> str:
    resp = cfn.describe_stacks(StackName=stack_name)
    for o in resp["Stacks"][0].get("Outputs", []):
        if o["OutputKey"] == key:
            return o["OutputValue"]
    return ""

BETA_API_URL = get_output(BETA_STACK, "ApiGatewayUrl")
PROD_API_URL = get_output(PROD_STACK, "ApiGatewayUrl")
print(f"Beta API: {BETA_API_URL}")
print(f"Prod API: {PROD_API_URL}")

## 5. Deploy Lambda Handler Code

In [ ]:
import io, zipfile

HANDLER_CODE = '''
import boto3, json, os
from datetime import datetime, timezone

def lambda_handler(event, context):
    environment = os.environ.get("ENVIRONMENT", "beta")
    bucket = os.environ["S3_BUCKET_NAME"]
    body = json.loads(event.get("body") or "{}")
    text = body.get("text", "").strip()
    if not text:
        return {"statusCode": 400, "body": json.dumps({"error": "Missing text"})}
    polly = boto3.client("polly")
    resp  = polly.synthesize_speech(Text=text, OutputFormat="mp3", VoiceId="Joanna", Engine="neural")
    audio = resp["AudioStream"].read()
    key   = f"polly-audio/{environment}/{datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%SZ\')}.mp3"
    boto3.client("s3").put_object(Bucket=bucket, Key=key, Body=audio, ContentType="audio/mpeg")
    return {"statusCode": 200, "body": json.dumps({"message": "Audio synthesized", "s3_key": key})}
'''

buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("handler.py", HANDLER_CODE)
zip_bytes = buf.getvalue()

lambda_client = boto3.client("lambda", region_name=AWS_REGION)
for fn in ["PollyTextToSpeech_Beta", "PollyTextToSpeech_Prod"]:
    lambda_client.update_function_code(FunctionName=fn, ZipFile=zip_bytes)
    print(f"Code deployed to {fn}")

## 6. Invoke the Endpoint

In [ ]:
import json, urllib.request

# Switch between "beta" and "prod"
ENVIRONMENT = "beta"   # <-- change to "prod" for production

endpoint = BETA_API_URL if ENVIRONMENT == "beta" else PROD_API_URL
payload  = json.dumps({"text": "This is from a Lambda via SageMaker."}).encode()

req = urllib.request.Request(
    endpoint,
    data=payload,
    headers={"Content-Type": "application/json"},
    method="POST",
)

with urllib.request.urlopen(req) as resp:
    result = json.loads(resp.read())

print(f"Response: {result}")
print(f"Audio saved to: s3://{S3_BUCKET_NAME}/{result.get('s3_key', '')}")

## 7. Verify Output in S3

In [ ]:
s3 = boto3.client("s3", region_name=AWS_REGION)

for prefix in ["polly-audio/beta/", "polly-audio/prod/"]:
    resp  = s3.list_objects_v2(Bucket=S3_BUCKET_NAME, Prefix=prefix)
    files = [o["Key"] for o in resp.get("Contents", [])]
    print(f"{prefix}: {len(files)} file(s)")
    for f in files[-3:]:
        print(f"  {f}")

## 8. Cleanup (optional)

Delete the CloudFormation stacks to remove all provisioned resources.

In [ ]:
# Uncomment to delete stacks
# for stack in [BETA_STACK, PROD_STACK]:
#     cfn.delete_stack(StackName=stack)
#     print(f"Deleting {stack}...")